# Fine-tune an Intent Router (Qwen3) — SNIPS dataset variant

**Recipe-generalization test for the Week 5 project.** Run this on a **free T4 GPU
runtime** — both Google Colab and Kaggle Notebooks work (Section 1 auto-detects which
one you're on). On Kaggle, enable Settings > Accelerator > GPU and Settings > Internet >
On before running.

This is the exact same recipe as
[`finetune_family_request_router.ipynb`](finetune_family_request_router.ipynb) — fine-tune
`Qwen/Qwen3-1.7B-Base` with a LoRA adapter through the
[LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory) visual UI — applied to a second,
unrelated dataset instead of the family-calendar routing task, as a check that the recipe's
results (and the classification-report / confusion-matrix workflow) generalize rather than
being an artifact of one specific dataset.

Dataset: `data/snips_intent_routing.csv` in this repo — a stratified subsample of the
[SNIPS NLU benchmark](https://github.com/sonos/nlu-benchmark) (via the
[`benayas/snips`](https://huggingface.co/datasets/benayas/snips) mirror), sized to roughly
match `family_request_routing.csv`'s per-label row count. See
`tools/prepare_snips_dataset.py` and `data/snips_intent_routing_manifest.json` for exact
provenance.

---
## The scenario

This notebook swaps the domain but keeps the problem shape identical to the family-request
router: given one short natural-language utterance, predict which of **seven** intents it
belongs to, then hand off to whatever downstream handler owns that intent. Here the
utterances are SNIPS's classic voice-assistant commands instead of parent requests to a
family-calendar coordinator:

```
AddToPlaylist         -> add a named song/artist/album to an existing playlist
BookRestaurant        -> reserve a table/restaurant
GetWeather            -> current conditions or forecast for a place/time
PlayMusic             -> play a song/artist/album/genre now (no playlist target)
RateBook              -> state a rating for a named book or saga
SearchCreativeWork    -> look up a specific named creative work by title
SearchScreeningEvent  -> find movie showtimes/theaters/screenings
```

Why this dataset specifically: it has the same 7-label, short-utterance shape as the family
router, so a working recipe here is real evidence the recipe generalizes — rather than
picking an easier or differently-shaped task and calling that a win. `RateBook` vs.
`SearchCreativeWork` is this dataset's closest analogue to the family router's
`fast_path_reject` vs. `ambiguous_clarify` boundary: both are "about a named book," and the
only distinguishing signal is whether a rating value is actually stated.

## 1. Install dependencies

In [ ]:
import os
from pathlib import Path

# Portable working directory: Colab uses /content, Kaggle uses /kaggle/working
# (and needs Settings -> Internet: On and Settings -> Accelerator: GPU set
# explicitly). Every path below is built from WORKDIR instead of a hardcoded
# platform-specific path, so the rest of this notebook runs unchanged on
# either platform.
if Path("/content").exists():
    WORKDIR = "/content"
elif Path("/kaggle/working").exists():
    WORKDIR = "/kaggle/working"
else:
    WORKDIR = str(Path.cwd())

os.environ["WORKDIR"] = WORKDIR
print(f"Detected environment -> WORKDIR = {WORKDIR}")

In [ ]:
%cd $WORKDIR
!rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print(
      "Please set up a GPU before using LLaMA Factory: on Colab, "
      "Runtime > Change runtime type > T4 GPU; on Kaggle, "
      "Settings > Accelerator > GPU (and Settings > Internet > On)."
  )

## 2. Prepare the SNIPS intent-routing dataset (INPUT REQUIRED)

The labelled CSV already lives in this repo, so the simplest path is to clone it straight
into this runtime. If the repo isn't reachable (private, not pushed yet, no internet access
enabled, etc.), this falls back to a manual upload widget for `snips_intent_routing.csv` on
Colab — on Kaggle, use "Add Input" to attach the CSV instead and set `CSV_PATH` manually.

This cell filters to the seven known labels, creates a **stratified 80/20 train/val split**,
converts the training split to **ShareGPT JSON**, writes it to `SNIPS_TRAIN.json` in the
LLaMA-Factory data dir, and registers it in `dataset_info.json` under the name
`snips_intent_routing` so it shows up in the LLaMA Board UI.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# ── constants ────────────────────────────────────────────────────────────────
# WORKDIR comes from the environment-detection cell in Section 1.
LLAMA_DATA_DIR  = f"{WORKDIR}/LLaMA-Factory/data"
TRAIN_JSON_PATH = f"{LLAMA_DATA_DIR}/SNIPS_TRAIN.json"
DATASET_INFO    = f"{LLAMA_DATA_DIR}/dataset_info.json"
REPO_URL        = "https://github.com/anushaakkiraju26/family-request-router.git"
# This dataset trial lives on its own branch, kept separate from main until
# it's reviewed and merged — clone that branch specifically, not the default.
REPO_BRANCH     = "data/snips-intent-trial"
REPO_DIR        = f"{WORKDIR}/family-request-router"

LABEL2ID = {
    "AddToPlaylist":        0,
    "BookRestaurant":       1,
    "GetWeather":           2,
    "PlayMusic":            3,
    "RateBook":             4,
    "SearchCreativeWork":   5,
    "SearchScreeningEvent": 6,
}

# Single source of truth for label strings + their human-readable names —
# every later cell (baseline eval, classify(), confusion matrix, charts)
# reads LABEL_TOKENS / LABEL_DISPLAY / display_labels from here rather than
# redeclaring them, so there's one place to edit if labels ever change.
LABEL_TOKENS = list(LABEL2ID)
LABEL_DISPLAY = {
    "AddToPlaylist":        "Add to playlist",
    "BookRestaurant":       "Book restaurant",
    "GetWeather":           "Get weather",
    "PlayMusic":            "Play music",
    "RateBook":             "Rate book",
    "SearchCreativeWork":   "Search creative work",
    "SearchScreeningEvent": "Search screening event",
}
display_labels = [LABEL_DISPLAY[l] for l in LABEL_TOKENS]

# System prompt — same discipline as the family router's: explicit
# disambiguating detail per label, calling out the pairs most likely to be
# confused (RateBook/SearchCreativeWork, PlayMusic/AddToPlaylist). This exact
# string must also be used at inference time in cell 16's classify().
SYSTEM_PROMPT = (
    "You are an intent-classification assistant for a voice assistant. Given "
    "a user utterance, respond with exactly one of the following seven "
    "categories:\n"
    "- AddToPlaylist: add a named song, artist, or album to an existing or "
    "named playlist. Requires a playlist target.\n"
    "- BookRestaurant: reserve a table or make a restaurant booking, "
    "typically naming a restaurant, party size, date/time, or "
    "cuisine/location.\n"
    "- GetWeather: ask for current conditions or a forecast for a place "
    "and/or time.\n"
    "- PlayMusic: request that a song, artist, album, genre, or playlist be "
    "played right now. Use this rather than AddToPlaylist when nothing is "
    "being added to a playlist.\n"
    "- RateBook: give or state a rating (a number of stars/points) for a "
    "named book or saga. Use this ONLY when a rating value is actually "
    "stated — not when merely searching for or referring to the book.\n"
    "- SearchCreativeWork: look up or find a specific named creative work "
    "(movie, book, game, song, TV show) by title, with no rating given and "
    "no reference to a screening or showtime.\n"
    "- SearchScreeningEvent: find movie showtimes, theaters, or screenings "
    "for a film, typically naming a time or theater.\n"
    "Respond with exactly one label and nothing else."
)

# ── 1. Get the CSV: clone this repo's branch, or fall back to manual upload ──
csv_path = Path(REPO_DIR) / "data" / "snips_intent_routing.csv"
if not csv_path.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        check=False,
    )

if csv_path.exists():
    CSV_PATH = str(csv_path)
    print(f"Using cloned dataset: {CSV_PATH}")
else:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError(
            "Could not clone the dataset repo and no Colab upload widget is "
            "available in this environment. On Kaggle: use 'Add Input' to "
            "attach snips_intent_routing.csv, then set CSV_PATH to its path "
            "(usually /kaggle/input/<dataset-name>/snips_intent_routing.csv) "
            "and re-run from the next cell."
        )
    print("Repo clone unavailable — upload snips_intent_routing.csv manually:")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]

# ── 2. Load + filter ────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH).rename(columns={"category_truth": "label"})
df = df[df["label"].isin(LABEL2ID)].sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Loaded {len(df):,} rows")
print(df["label"].value_counts())

# ── 3. Stratified train/val split ───────────────────────────────────────────
df_train, df_val = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42,
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
print(f"\nTrain: {len(df_train):,} rows | Val (held-out): {len(df_val):,} rows")
print("\nTrain label distribution:")
print(df_train["label"].value_counts())

# ── 4. Convert train split -> ShareGPT JSON ─────────────────────────────────
sharegpt_records = [
    {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Utterance: {row['text']}"},
            {"role": "assistant", "content": row["label"]},
        ]
    }
    for _, row in df_train.iterrows()
]

with open(TRAIN_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(sharegpt_records, f, indent=2, ensure_ascii=False)
print(f"\nWritten {len(sharegpt_records):,} records -> {TRAIN_JSON_PATH}")

# ── 5. Register dataset in dataset_info.json ────────────────────────────────
with open(DATASET_INFO, "r", encoding="utf-8") as f:
    info = json.load(f)

info["snips_intent_routing"] = {
    "file_name": "SNIPS_TRAIN.json",
    "formatting": "sharegpt",
    "columns":   {"messages": "messages"},
    "tags": {
        "role_tag":      "role",
        "content_tag":   "content",
        "user_tag":      "user",
        "assistant_tag": "assistant",
        "system_tag":    "system",
    },
}

with open(DATASET_INFO, "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)
print(f"Registered 'snips_intent_routing' in {DATASET_INFO}")

# ── 6. Save val split for evaluation ────────────────────────────────────────
VAL_CSV = f"{WORKDIR}/snips_val_split.csv"
df_val.to_csv(VAL_CSV, index=False)
print(f"Val split saved -> {VAL_CSV}  ({len(df_val):,} rows)")

## 3. Fine-tune via LLaMA Board (INPUT REQUIRED)

1. Run the next cell to start the LLaMA Board server, then open the **public** URL it prints.
2. Base model: `Qwen/Qwen3-1.7B-Base`. Dataset: `snips_intent_routing`. Finetuning: **LoRA**.
   Defaults are fine for a first run.
3. When training finishes, note the **Output Dir** path (`train_2026-...`) — you'll need it below.
4. **Manually stop this cell** once training completes — neither Colab nor Kaggle will stop
   the server for you.

Watch for the "training completed" message in the UI or the logs. Occasional "syntax error"
toasts from LLaMA Board are a known cosmetic issue and don't affect training. A short LoRA run
on a few hundred training rows usually takes well under the reference project's 30–60 min
estimate.

### Hyperparameters, briefly

Defaults are fine for a first pass. If you need to adjust:

| Knob | Effect | If your run is off |
|---|---|---|
| Learning rate | step size per update | loss oscillating -> lower it; loss barely moving -> raise it carefully |
| Epochs | passes over training data | loss still falling at the end -> add an epoch; val score drops late -> stop earlier |
| Batch size | examples per gradient step | tune for GPU memory / gradient noise |
| LoRA rank | adapter capacity | raise only if the task clearly needs more expressiveness — most of these seven labels don't, but the RateBook/SearchCreativeWork boundary might |

In [ ]:
%cd $WORKDIR/LLaMA-Factory
!GRADIO_SHARE=1 llamafactory-cli webui

---
## 4. Review training — loss curve (INPUT REQUIRED)

Set `ADAPTER_DIR` to the Output Dir path from LLaMA Board. The loss should drop and level
off; flat or chaotic usually means the learning rate, dataset size, or chat template is off.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

ADAPTER_DIR = f"{WORKDIR}/LLaMA-Factory/saves/Qwen3-1.7B-Base/lora/train_XXXX-XX-XX-XX-XX-XX"  # <-- CHANGE THIS
MERGED_DIR      = f"{WORKDIR}/snips_router_merged"
BASE_MODEL_NAME = "Qwen/Qwen3-1.7B-Base"

log_file = Path(ADAPTER_DIR) / "trainer_log.jsonl"
if not log_file.exists():
    raise FileNotFoundError(f"trainer_log.jsonl not found in {ADAPTER_DIR!r}")

records = [json.loads(l) for l in log_file.read_text().splitlines() if l.strip()]
steps   = [r["current_steps"] for r in records if r.get("loss") is not None]
losses  = [r["loss"]          for r in records if r.get("loss") is not None]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, linewidth=1.5, color="#1565C0", alpha=0.85)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss curve")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{WORKDIR}/training_curve.png", dpi=120)
plt.show()

total_drop = losses[0] - losses[-1]
print(f"Starting loss : {losses[0]:.4f}")
print(f"Final loss    : {losses[-1]:.4f}")
print(f"Total drop    : {total_drop:.4f}")
print()
if losses[-1] < 0.5:
    print("Loss is low - model has likely converged well.")
elif losses[-1] < 1.2:
    print("Loss is moderate - model has learned but may benefit from more epochs.")
else:
    print("Loss is still high - consider more epochs, a lower learning rate, or checking the data format.")

---
## 5. Merge the adapter and measure a baseline

Training kept the base weights frozen and only trained a small LoRA adapter. Merging folds
those deltas into the base weights, giving one standalone model with no adapter overhead.

The baseline below is the *same* base model, on the *same* validation set, with **no**
fine-tuning — a constrained seven-letter multiple-choice prompt so it can't fail just because
it phrases a label slightly differently. We measure it now, before attaching the adapter, so
the base model doesn't need to be loaded twice.

In [ ]:
import gc
import os
from pathlib import Path

import pandas as pd
import safetensors.torch as st
import torch
from peft import PeftConfig, get_peft_model
from peft.utils import set_peft_model_state_dict
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── USER INPUT ───────────────────────────────────────────────────────────────
# LABEL_TOKENS / LABEL_DISPLAY / display_labels / VAL_CSV come from cell 7.
CHOICES      = "ABCDEFG"
CHOICE2LABEL = {ch: lbl for ch, lbl in zip(CHOICES, LABEL_TOKENS)}

BASE_SYSTEM_PROMPT = (
    "You are an intent-classification assistant for a voice assistant. "
    "Classify the utterance by responding with ONLY a single letter - nothing else:\n"
    + "\n".join(f"{ch}) {lbl}" for ch, lbl in CHOICE2LABEL.items())
)

# ── Validate adapter ─────────────────────────────────────────────────────────
adapter_path = Path(ADAPTER_DIR)
if not adapter_path.is_dir():
    raise FileNotFoundError(f"Adapter folder not found: {ADAPTER_DIR!r}")
if not (adapter_path / "adapter_config.json").exists():
    raise FileNotFoundError(f"No adapter_config.json in {ADAPTER_DIR!r}")
print(f"Adapter found: {ADAPTER_DIR}")

# ── Device ───────────────────────────────────────────────────────────────────
HAS_CUDA = torch.cuda.is_available()
HAS_MPS  = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE   = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")
dtype    = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32
print(f"Device: {DEVICE}")

# ── Load base model + tokenizer ─────────────────────────────────────────────
print("\nLoading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=dtype, device_map=DEVICE,
)
tok_src    = str(adapter_path) if (adapter_path / "tokenizer.json").exists() else BASE_MODEL_NAME
_tokenizer = AutoTokenizer.from_pretrained(tok_src)
base_model.eval()

# ── Baseline inference on val split ─────────────────────────────────────────
_choice_ids = []
for ch in CHOICES:
    ids_plain  = _tokenizer.encode(ch,       add_special_tokens=False)
    ids_spaced = _tokenizer.encode(f" {ch}", add_special_tokens=False)
    _choice_ids.append(ids_spaced[0] if len(ids_spaced) == 1 else ids_plain[0])


def classify_base(request_text: str) -> str:
    messages = [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Utterance: {request_text}"},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model(input_ids=inputs["input_ids"])
    first_logits = out.logits[0, -1, :]
    choice_probs = torch.softmax(first_logits[_choice_ids], dim=-1).cpu().tolist()
    return CHOICE2LABEL[CHOICES[choice_probs.index(max(choice_probs))]]


df_val      = pd.read_csv(VAL_CSV)
y_true      = df_val["label"].tolist()
y_pred_base = []

for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Baseline inference"):
    y_pred_base.append(classify_base(row["text"]))

from sklearn.metrics import classification_report
print("\n=== Baseline (no fine-tuning) ===")
# labels=LABEL_TOKENS is required here: without it, sklearn silently sorts
# the true class strings alphabetically and zips target_names to THAT order
# instead of LABEL_TOKENS's order, printing every row under the wrong name
# (right numbers, wrong label) while looking completely normal.
print(classification_report(y_true, y_pred_base, labels=LABEL_TOKENS, target_names=display_labels, digits=3, zero_division=0))

# ── Attach LoRA + load weights ───────────────────────────────────────────────
print("Attaching LoRA adapter...")
peft_cfg   = PeftConfig.from_pretrained(str(adapter_path))
peft_model = get_peft_model(base_model, peft_cfg)

candidates = [
    adapter_path / "adapter_model.safetensors",
    adapter_path / "adapters.safetensors",
    adapter_path / "adapter_model.bin",
]
weights_file = next((p for p in candidates if p.exists()), None)
if weights_file is None:
    raise FileNotFoundError(f"No adapter weights in {ADAPTER_DIR}")

if weights_file.suffix == ".safetensors":
    adapter_weights = st.load_file(str(weights_file), device=DEVICE)
else:
    adapter_weights = torch.load(str(weights_file), map_location=DEVICE)

set_peft_model_state_dict(peft_model, adapter_weights)

# ── Merge + save to disk ─────────────────────────────────────────────────────
print("Merging (this may take a while) ...")
_model = peft_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
_model.save_pretrained(MERGED_DIR)
_tokenizer.save_pretrained(MERGED_DIR)
stale = Path(MERGED_DIR) / "adapter_config.json"
if stale.exists():
    stale.unlink()
print(f"Saved -> {MERGED_DIR}")

del peft_model, base_model, adapter_weights
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_model.eval()
print("Model ready for inference.")

---
## 6. `classify()` and a smoke test

Chat-format the request with the same system prompt used in training, generate a few tokens,
and match the start of the output to one of the seven labels. `classify()` also reports a
confidence score — the softmax probability of the winning label's first token versus the
others. Seven obvious requests, one per class, should all route correctly before trusting
the full validation run.

In [ ]:
import torch

# Must exactly match the SYSTEM_PROMPT used to build the training data in
# cell 7.
SYSTEM_PROMPT = (
    "You are an intent-classification assistant for a voice assistant. Given "
    "a user utterance, respond with exactly one of the following seven "
    "categories:\n"
    "- AddToPlaylist: add a named song, artist, or album to an existing or "
    "named playlist. Requires a playlist target.\n"
    "- BookRestaurant: reserve a table or make a restaurant booking, "
    "typically naming a restaurant, party size, date/time, or "
    "cuisine/location.\n"
    "- GetWeather: ask for current conditions or a forecast for a place "
    "and/or time.\n"
    "- PlayMusic: request that a song, artist, album, genre, or playlist be "
    "played right now. Use this rather than AddToPlaylist when nothing is "
    "being added to a playlist.\n"
    "- RateBook: give or state a rating (a number of stars/points) for a "
    "named book or saga. Use this ONLY when a rating value is actually "
    "stated — not when merely searching for or referring to the book.\n"
    "- SearchCreativeWork: look up or find a specific named creative work "
    "(movie, book, game, song, TV show) by title, with no rating given and "
    "no reference to a screening or showtime.\n"
    "- SearchScreeningEvent: find movie showtimes, theaters, or screenings "
    "for a film, typically naming a time or theater.\n"
    "Respond with exactly one label and nothing else."
)
# LABEL_TOKENS / LABEL_DISPLAY / display_labels come from cell 7.


def classify(request_text: str, compute_confidence: bool = True) -> tuple:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Utterance: {request_text}"},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(prompt, return_tensors="pt").to(_model.device)

    with torch.no_grad():
        out = _model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=_tokenizer.eos_token_id,
        )

    generated = _tokenizer.decode(
        out.sequences[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

    matched = next((l for l in LABEL_TOKENS if generated.lower().startswith(l.lower())), None)
    if matched is None:
        matched = "SearchCreativeWork"

    if not compute_confidence or not out.scores:
        return matched, 1.0

    first_scores = out.scores[0][0]
    probs = torch.softmax(first_scores, dim=-1)
    confidence = probs.max().item()
    return matched, confidence


SMOKE_TESTS = [
    ("Add Tobias Sammet to my bring back the 90s list", "AddToPlaylist"),
    ("Make me a reservation in Colorado at nine am at National Cash Register Building", "BookRestaurant"),
    ("What will the weather be in Allenwood ?", "GetWeather"),
    ("Play me some grunge music", "PlayMusic"),
    ("Rate the current textbook 4 out of 6 points", "RateBook"),
    ("Find a show called Some Kind of Dangerous .", "SearchCreativeWork"),
    ("What movies are playing nearby ?", "SearchScreeningEvent"),
]

correct = 0
for text, expected in SMOKE_TESTS:
    pred, conf = classify(text)
    ok = "OK" if pred == expected else "MISMATCH"
    correct += pred == expected
    print(f"[{ok:8s}] expected={expected:22s} predicted={pred:22s} conf={conf:.1%}  | {text}")

print(f"\n{correct}/{len(SMOKE_TESTS)} smoke tests passed.")

---
## 7. Evaluate on the held-out validation split

These rows were never seen during training, so this is the honest read on routing accuracy.

In [ ]:
from tqdm.auto import tqdm

y_pred = []
for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Fine-tuned inference"):
    pred, _ = classify(row["text"], compute_confidence=False)
    y_pred.append(pred)

print(f"Evaluated {len(y_pred):,} samples.")

In [ ]:
from sklearn.metrics import classification_report

# labels=LABEL_TOKENS is required: without it, sklearn silently sorts the
# true class strings alphabetically and zips target_names to THAT order
# instead of LABEL_TOKENS's order, printing every row under the wrong name.
print(classification_report(y_true, y_pred, labels=LABEL_TOKENS, target_names=display_labels, digits=3))

print()
for i in [0, 5, 10, 15]:
    if i >= len(df_val):
        continue
    row  = df_val.iloc[i]
    pred, conf = classify(row["text"])
    print(f"=== Request {i}")
    print(f"TRUE label: {row['label']}")
    print(f"PRED label: {pred}  (conf {conf:.1%})")
    print(f"Text: {row['text']}")
    print()

### Reading the report

There's no safety-critical label here the way `fast_path_reject`/`fast_path_mutate` were for
the family router — a routing mistake just sends the utterance to the wrong handler, not past
a safeguard. The label most worth watching is `RateBook`: it's this dataset's tightest
minimal-pair boundary against `SearchCreativeWork` (both are "about a named book," and the
only distinguishing signal is whether a rating value is actually stated), which makes it the
closest analogue to the family router's `fast_path_reject`/`ambiguous_clarify` confusion.

### Confusion matrix

Rows are true labels, columns are predictions; the diagonal is correct. Colour is
row-normalised so it doubles as a recall heatmap. Watch for `RateBook` rows leaking into
`SearchCreativeWork` (or the reverse) — that's the confusion pair worth the most attention,
and the one to compare against the family router's own hardest boundary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm      = confusion_matrix(y_true, y_pred, labels=LABEL_TOKENS)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm, annot=cm, fmt="d", cmap="Blues",
    xticklabels=display_labels, yticklabels=display_labels,
    linewidths=0.5, ax=ax,
)
ax.set_title("Confusion matrix (counts shown, colour = row-normalised recall)")
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{WORKDIR}/confusion_matrix.png", dpi=120)
plt.show()

ratebook_recall = cm_norm[LABEL_TOKENS.index("RateBook"), LABEL_TOKENS.index("RateBook")]
print(f"\nRateBook recall: {ratebook_recall:.1%}")
if ratebook_recall < 0.85:
    print("Warning: RateBook recall below 0.85 - likely leaking into SearchCreativeWork.")

search_creative_recall = cm_norm[LABEL_TOKENS.index("SearchCreativeWork"), LABEL_TOKENS.index("SearchCreativeWork")]
print(f"SearchCreativeWork recall: {search_creative_recall:.1%}")
if search_creative_recall < 0.85:
    print("Warning: SearchCreativeWork recall below 0.85 - likely leaking into RateBook.")

---
## 8. Baseline vs fine-tuned

The baseline is the identical base model, same validation requests, zero task-specific
training — its best shot via the constrained seven-letter prompt. The gap between it and the
fine-tuned model is the measurable value of this labelled dataset and this training run.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report


def per_class_f1(y_t, y_p, labels):
    # labels=labels (sklearn's own ordering param) is required: without it,
    # sklearn silently sorts the true class strings alphabetically and zips
    # target_names to THAT order instead of `labels`'s order, so report[lbl]
    # below would silently fetch another class's f1-score.
    report = classification_report(y_t, y_p, labels=labels, target_names=labels, output_dict=True, zero_division=0)
    return {lbl: report[lbl]["f1-score"] for lbl in labels}


ft_f1    = per_class_f1(y_true, y_pred,      LABEL_TOKENS)
base_f1  = per_class_f1(y_true, y_pred_base, LABEL_TOKENS)
ft_acc   = accuracy_score(y_true, y_pred)
base_acc = accuracy_score(y_true, y_pred_base)

labels_plot = display_labels + ["overall accuracy"]
ft_vals     = [ft_f1[l]   for l in LABEL_TOKENS] + [ft_acc]
base_vals   = [base_f1[l] for l in LABEL_TOKENS] + [base_acc]

x     = np.arange(len(labels_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_base = ax.bar(x - width/2, base_vals, width, label="Base model (no fine-tuning)", color="#90CAF9", edgecolor="white")
bars_ft   = ax.bar(x + width/2, ft_vals,   width, label="Fine-tuned (LLaMA Board LoRA)", color="#1565C0", edgecolor="white")

ax.bar_label(bars_base, fmt="{:.2f}", padding=3, fontsize=8)
ax.bar_label(bars_ft,   fmt="{:.2f}", padding=3, fontsize=8)
bars_base[-1].set_color("#FFCC80")
bars_ft[-1].set_color("#E65100")

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels([l.replace(" ", "\n") for l in labels_plot], fontsize=9)
ax.set_ylabel("F1 score / Accuracy")
ax.set_title("SNIPS intent router - baseline vs fine-tuned")
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.15), ncol=2)
plt.tight_layout()
plt.savefig(f"{WORKDIR}/baseline_vs_finetuned.png", dpi=120)
plt.show()

print(f"Baseline accuracy    : {base_acc:.1%}")
print(f"Fine-tuned accuracy  : {ft_acc:.1%}")
print(f"Delta                : {(ft_acc - base_acc) * 100:+.1f} pts")

## Recap

- Fine-tuned `Qwen/Qwen3-1.7B-Base` with a LoRA adapter through LLaMA Board on
  `data/snips_intent_routing.csv` — a stratified subsample of the SNIPS NLU benchmark
  (7 intents, ~455 rows total), prepared by `tools/prepare_snips_dataset.py` to roughly match
  `family_request_routing.csv`'s per-label row count.
- Compared the merged fine-tuned model against the same constrained-choice baseline harness
  used in the family-router notebook, on a held-out validation split.
- **Purpose of this run**: not to build a production SNIPS classifier, but to check whether
  the family-router notebook's recipe — same base model, same LoRA settings, same
  baseline-vs-fine-tuned evaluation harness — produces a similarly large accuracy gain on an
  unrelated, well-known 7-intent task. A comparable gain here is evidence the recipe itself
  generalizes; a much smaller or larger one would say the family-router's own results are more
  (or less) dataset-specific than they look in isolation.

### Results — fill in after running

| Metric | Baseline | Fine-tuned |
|---|---|---|
| Overall accuracy | _fill in_ | _fill in_ |

Per-class recall (fine-tuned):

| Label | Recall |
|---|---|
| `AddToPlaylist` | _fill in_ |
| `BookRestaurant` | _fill in_ |
| `GetWeather` | _fill in_ |
| `PlayMusic` | _fill in_ |
| `RateBook` | _fill in_ |
| `SearchCreativeWork` | _fill in_ |
| `SearchScreeningEvent` | _fill in_ |

### Comparison to the family-router recipe

Once this run finishes, compare its overall-accuracy delta (fine-tuned minus baseline) and
its `RateBook`/`SearchCreativeWork` confusion (this dataset's closest analogue to the family
router's `fast_path_reject`/`ambiguous_clarify` boundary) against the family-router
notebook's own Recap section — same base model and LoRA settings, different data.